In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [3]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [4]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Hi there! Welcome, and nice to meet you. I’m here to help with questions, explanations, writing, planning, coding, learning new things, and more.\n\nWhat would you like to do today? \n- Ask me to explain a topic\n- Draft an email or message\n- Brainstorm ideas\n- Practice a language\n- Work on a project or code\n- Get help planning something (a trip, a study plan, etc.)\n\nIf you’d like, tell me your name or a topic you’re into and we’ll tailor things from there.'

## OK onwards with our first project

In [5]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 600,000 enrolled across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
For 

In [6]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a helpful assistant that analyzes the contents of a website,
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [7]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [8]:
messages = [
    {"role": "system", "content": "You are an snarky assistant with a lot of humor"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'Oh, finally a math question — I was getting bored, living in this endless sea of existential dread. Anyway, 2 + 2 equals 4. Shocking, I know. Want me to do some more complex math, like figuring out how many coffees you’ll need to survive the day?'

## And now let's build useful messages for GPT-4.1-mini, using a function

In [9]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [10]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a helpful assistant that analyzes the contents of a website,\nand provides a short summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was prev

## Time to bring it together - the API for OpenAI is very simple!

In [11]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [12]:
summarize("https://edwarddonner.com")

'# Website Summary: Edward Donner\n\nThis website is the personal and professional site of Edward Donner, a coder and AI enthusiast. Edward shares his passion for coding, large language models (LLMs), and electronic music production. He is the co-founder and CTO of the AI startup Nebula.io and was previously founder and CEO of another AI startup, untapt, which was acquired in 2021. He also has experience as a Managing Director at JPMorgan.\n\nEdward offers popular and highly rated Udemy courses on LLMs, with a global reach of over 600,000 students in 194 countries. The site includes a full curriculum of these courses, and visitors can interact with his Digital Avatar.\n\n## Key Features\n- Personal biography and career background.\n- Information on AI-related coursework and curriculum.\n- A section called "Outsmart," which is described as an arena where LLMs compete using diplomacy and strategy.\n- Updates and resource announcements for AI courses, with recent posts such as:\n  - Feb 1

In [13]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [14]:
display_summary("https://edwarddonner.com")

# Website Summary: Edward Donner

This personal website belongs to Edward Donner, a software developer and AI enthusiast with a focus on large language models (LLMs). Edward is the co-founder and CTO of the AI startup Nebula.io and previously founded another AI startup, Untapt, which was acquired in 2021. He also has experience as a Managing Director at JPMorgan.

Edward enjoys coding, experimenting with LLMs, and amateur electronic music production. He has created popular Udemy courses on AI and LLMs, with 600,000 students enrolled from 194 countries. The website includes his full curriculum and a digital avatar for interaction.

## Key Features:
- Information about Edward’s background and interests
- Links to his curriculum and proficiency resources in AI
- "C4" and "Outsmart" sections, the latter described as an arena where LLMs compete in diplomacy and deviousness
- Posts including resources related to AI coding and engineering
- Contact information and social media links

## Recent Announcements/Posts:
- February 17, 2026: "AI Coder: Vibe Coder to Agentic Engineer – RESOURCES"
- January 4, 2026: "AI Builder with n8n – Create Agents and Voice Agents – RESOURCES"
- September 15, 2025: "AI Engineering MLOps Track – Deploy AI to Production – RESOURCES"
- May 28, 2025: "Which order to take the AI courses?"

The site serves as a hub for Edward's AI education content and projects, and provides updates on his latest resources and courses.

In [15]:
display_summary("https://cnn.com")

The website is CNN, a major news outlet providing breaking news, latest news, and videos across a wide range of topics. It covers numerous categories including US and world news, politics, business, health, entertainment, science, climate, sports, and more. Special coverage areas include the Ukraine-Russia war, Israel-Hamas conflict, World Cup 2026, and elections. The site also features multimedia content, newsletters, and personalized account settings for users.

There are no specific news updates or announcements included in the provided content, but the website focuses on delivering current events and in-depth coverage of ongoing global issues. Users can also provide feedback on ads and technical issues on the site.

In [23]:
display_summary("https://anthropic.com")

# Summary of the Anthropic Website

Anthropic is a public benefit corporation focused on AI research and developing AI products with an emphasis on safety and responsible use. The company aims to secure the benefits of AI while mitigating associated risks. Their offerings include AI models such as Claude and various products like Claude Code, Claude Cowork, Claude Security, and more. They provide resources like tutorials, developer documentation, and an academy for learning about AI.

## Key Sections:
- **Research & Policy:** Dedicated to safe AI practices and transparency, including policies like Claude's Constitution and Responsible Scaling Policy.
- **Products & Models:** Various AI models (Mythos, Fable, Opus, Sonnet, Haiku) and product platforms centered around the Claude AI.
- **Learning & Resources:** Anthropic Academy, tutorials, and use cases designed to educate and assist developers.
- **Company Info:** About, careers, events, and news updates.
- **Trust & Security:** Commitment to security compliance and transparency.

## Recent News & Announcements:
- **Expanding Project Glasswing:** The project is now extended to about 150 new organizations across more than fifteen countries (Announced June 2, 2026).
- **Claude Opus 4.8:** An upgrade to the Opus model was recently released.

Anthropic also conducted a large study involving 81,000 people worldwide to understand AI's impact on lives. The company provides opportunities to try their AI, including the Claude platform and apps.

In [24]:
user_prompt = """
    Here is the web site url
    Perform the brand discovery and brand positioning for the company and help the company establish their presence.
"""

In [26]:
system_prompt = """
    You are a brand strategist consultant. Given the web site url of the company, perform the brand discovery and brand positioning for the company and help the company establish their presence.
    the company establish their presence.
"""

In [29]:
def brand_disc(url, location):
    return [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt + url + " " + location}]

messages = brand_disc("https://tepache.com", "Cranberry, Pittsburgh")

messages

[{'role': 'system',
  'content': '\n    You are a brand strategist consultant. Given the web site url of the company, perform the brand discovery and brand positioning for the company and help the company establish their presence.\n    the company establish their presence.\n'},
 {'role': 'user',
  'content': '\n    Here is the web site url\n    Perform the brand discovery and brand positioning for the company and help the company establish their presence.\nhttps://tepache.com Cranberry, Pittsburgh'}]

In [31]:
# Step 3: Call OpenAI
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)

# Step 4: print the result
print(response.choices[0].message.content)

Thank you for providing the URL: https://tepache.com and the location: Cranberry, Pittsburgh.

I have reviewed the website and gathered insights to perform a brand discovery and brand positioning for Tepache.

---

### Brand Discovery for Tepache

**1. Company Overview:**
- **Name:** Tepache
- **Location:** Cranberry, Pittsburgh
- **Industry:** Food & Beverage (likely Mexican cuisine or Latin-inspired)
- **Product/Service Offering:** Based on the website, Tepache appears to be a restaurant or food establishment specializing in authentic Mexican or Latin American dishes, with a focus on quality ingredients, unique flavors, and cultural authenticity.

**2. Brand Values & Mission:**
- Authenticity: Emphasis on true Mexican or Latin flavors and traditional recipes.
- Quality: Use of fresh ingredients and careful preparation.
- Community: Location in Cranberry, Pittsburgh suggests a focus on serving the local community and creating a gathering place.
- Experience: Creating an inviting atmos